In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/santoshd3/crop-price-prediction/dataset_42_soybean.csv
/kaggle/input/datasets/santoshd3/crop-price-prediction/corn yield.csv
/kaggle/input/datasets/jignalgajjar/daily-crop-mandi-price-with-weather-data2023-25/master_aggriculture_dataset.csv
/kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/generation_methodology.md
/kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/data_dictionary.csv
/kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/README.md
/kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/bharatbench_agriculture.csv
/kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/CHANGELOG.md
/kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/quality_report.md
/kaggle/input/datasets/iconicwasil/us-crop-prices-and-midwest-climate-data-2010-2024/US_Agriculture_Weather_2010_2024.csv
/kaggle/input/datas

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# ૧. CSV ફાઇલ શોધો અને લોડ કરો
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
print("મળેલ CSV ફાઇલો:")
for i, f in enumerate(csv_files):
    print(f"[{i}] {f}")

# 'Crop Price' અથવા પહેલી CSV ફાઇલ લોડ કરો
selected_file = [f for f in csv_files if 'price' in f.lower()][0] if csv_files else csv_files[0]
print(f"\n📂 વાપરવામાં આવતી ફાઇલ: {selected_file}")

df = pd.read_csv(selected_file)
print("\n--- ડેટાસેટના પ્રથમ 5 રેકોર્ડ્સ ---")
print(df.head())

મળેલ CSV ફાઇલો:
[0] /kaggle/input/datasets/santoshd3/crop-price-prediction/dataset_42_soybean.csv
[1] /kaggle/input/datasets/santoshd3/crop-price-prediction/corn yield.csv
[2] /kaggle/input/datasets/jignalgajjar/daily-crop-mandi-price-with-weather-data2023-25/master_aggriculture_dataset.csv
[3] /kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/data_dictionary.csv
[4] /kaggle/input/datasets/mansiaggarwal88/bharatbench-indian-agriculture-benchmark/bharatbench_agriculture.csv
[5] /kaggle/input/datasets/iconicwasil/us-crop-prices-and-midwest-climate-data-2010-2024/US_Agriculture_Weather_2010_2024.csv
[6] /kaggle/input/datasets/jyvastani/price-predictor/9ef84268-d588-465a-a308-a864a43d0070.csv

📂 વાપરવામાં આવતી ફાઇલ: /kaggle/input/datasets/santoshd3/crop-price-prediction/dataset_42_soybean.csv

--- ડેટાસેટના પ્રથમ 5 રેકોર્ડ્સ ---
      date plant-stand    precip   temp  hail          crop-hist area-damaged  \
0  october      normal   gt-norm   norm   yes       

In [3]:
# ૨. ખાલી વેલ્યુ સાફ કરો
df = df.dropna()

# કઈ કોલમ ભાવ (Price) છે તે શોધો
price_cols = [c for c in df.columns if 'price' in c.lower() or 'modal' in c.lower()]
target_col = price_cols[0] if price_cols else df.columns[-1]
print(f"🎯 Target Column (ભાવ): {target_col}")

# ૩. ડેટાને Numbers માં કન્વર્ટ કરો (One-Hot Encoding)
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(columns=[c for c in df_encoded.columns if target_col in c])
y = df[target_col]

# 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ૪. Random Forest મોડેલ ટ્રેન કરો
print("\n⏳ મોડેલ ટ્રેનિંગ શરૂ થઈ રહ્યું છે...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ૫. Accuracy અને Error ચેક કરો
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n=================================")
print(f"✅ મોડેલ સફળતાપૂર્વક ટ્રેન થઈ ગયું!")
print(f"📊 Accuracy (R2 Score): {r2 * 100:.2f}%")
print(f"📉 Mean Absolute Error (MAE): ₹{mae:.2f}")
print("=================================")

# ૬. મોડેલ સેવ કરો
joblib.dump(model, '/kaggle/working/crop_price_model.pkl')
joblib.dump(X.columns.tolist(), '/kaggle/working/model_features.pkl')
print("💾 Model saved to /kaggle/working/crop_price_model.pkl")

🎯 Target Column (ભાવ): class

⏳ મોડેલ ટ્રેનિંગ શરૂ થઈ રહ્યું છે...


ValueError: could not convert string to float: ' brown-spot'

In [ ]:
# ૨. ખાલી વેલ્યુ સાફ કરો
df = df.dropna()

# કઈ કોલમ ભાવ (Price) છે તે શોધો
price_cols = [c for c in df.columns if 'price' in c.lower() or 'modal' in c.lower()]
target_col = price_cols[0] if price_cols else df.columns[-1]
print(f"🎯 Target Column (ભાવ): {target_col}")

# ૩. ડેટાને Numbers માં કન્વર્ટ કરો (One-Hot Encoding)
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(columns=[c for c in df_encoded.columns if target_col in c])
y = df[target_col]

# 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ૪. Random Forest મોડેલ ટ્રેન કરો
print("\n⏳ મોડેલ ટ્રેનિંગ શરૂ થઈ રહ્યું છે...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ૫. Accuracy અને Error ચેક કરો
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n=================================")
print(f"✅ મોડેલ સફળતાપૂર્વક ટ્રેન થઈ ગયું!")
print(f"📊 Accuracy (R2 Score): {r2 * 100:.2f}%")
print(f"📉 Mean Absolute Error (MAE): ₹{mae:.2f}")
print("=================================")

# ૬. મોડેલ સેવ કરો
joblib.dump(model, '/kaggle/working/crop_price_model.pkl')
joblib.dump(X.columns.tolist(), '/kaggle/working/model_features.pkl')
print("💾 Model saved to /kaggle/working/crop_price_model.pkl")

In [ ]:
import glob
import pandas as pd

# બધી CSV ફાઇલો શોધો
all_csv = glob.glob('/kaggle/input/**/*.csv', recursive=True)

print("🔍 તમારા Datasets ની CSV ફાઇલો:")
for i, path in enumerate(all_csv):
    print(f"[{i}] {path}")

In [ ]:
import numpy as np
import pandas as pd
import glob
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# ૧. સાચી Price વાળી ફાઇલ પસંદ કરો
all_csv = glob.glob('/kaggle/input/**/*.csv', recursive=True)

# 💡 જો તમને લિસ્ટમાં સાચો નંબર ખબર હોય તો અહીં 0 ની જગ્યાએ એ નંબર લખો (દા.ત. 0, 1, 2)
# અથવા આ લાઈન આપમેળે 'mandi' કે 'price' વાળી ફાઈલ શોધી લેશે:
price_files = [f for f in all_csv if 'disease' not in f.lower() and ('price' in f.lower() or 'mandi' in f.lower() or 'crop' in f.lower())]
target_file = price_files[0] if price_files else all_csv[0]

print(f"📂 લોડ કરેલ ફાઇલ: {target_file}")
df = pd.read_csv(target_file)

# ૨. કોલમ્સ જુઓ
print("\nડેટાસેટની Columns:", df.columns.tolist())
print(df.head())

# ૩. Missing Data સાફ કરો
df = df.dropna()

# ૪. ફક્ત ભાવ (Numeric Price) વાળી કોલમ શોધો
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
price_cols = [c for c in numeric_cols if 'price' in c.lower() or 'modal' in c.lower() or 'msp' in c.lower()]

if price_cols:
    target_col = price_cols[0]
else:
    # જો price શબ્દ ન હોય તો છેલ્લી numeric કોલમ લો
    target_col = numeric_cols[-1]

print(f"\n🎯 Target Column (ભાવ - Price): {target_col}")

# ૫. Features (X) અને Target (y)
y = df[target_col]
X_raw = df.drop(columns=[target_col])

# Categorical ટેક્સ્ટને Numbers માં ફેરવો (One-Hot Encoding)
X = pd.get_dummies(X_raw, drop_first=True)

# ૬. 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ૭. Random Forest Model Train કરો
print("\n⏳ મોડેલ ટ્રેનિંગ શરૂ થઈ રહ્યું છે...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ૮. Accuracy ચેક કરો
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n=================================")
print("✅ મોડેલ સફળતાપૂર્વક ટ્રેન થઈ ગયું!")
print(f"📊 Accuracy (R2 Score): {r2 * 100:.2f}%")
print(f"📉 Mean Error (MAE): ₹{mae:.2f}")
print("=================================")

# ૯. Model Save કરો
joblib.dump(model, '/kaggle/working/crop_price_model.pkl')
joblib.dump(X.columns.tolist(), '/kaggle/working/model_features.pkl')
print("💾 Model saved to /kaggle/working/crop_price_model.pkl")

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# ૧. સાચી Price / Mandi વાળી CSV શોધો (BharatBench સિવાયની)
all_csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)

# ફક્ત Price વાળા ફોલ્ડર્સમાંથી CSV ફિલ્ટર કરો
price_csvs = [f for f in all_csvs if 'bharatbench' not in f.lower() and ('price' in f.lower() or 'mandi' in f.lower())]

if not price_csvs:
    # જો નામ ન મળે તો BharatBench સિવાયની કોઈ પણ CSV લો
    price_csvs = [f for f in all_csvs if 'bharatbench' not in f.lower()]

selected_file = price_csvs[0]
print(f"✅ સાચો ડેટાસેટ સિલેક્ટ થયો: {selected_file}")

# ૨. Data Load કરો
df = pd.read_csv(selected_file)
print("\n--- ડેટાસેટના Columns ---")
print(df.columns.tolist())
print(df.head(3))

# ૩. Missing values સાફ કરો
df = df.dropna()

# ૪. ભાવ (Price) વાળી કોલમ શોધો
target_candidates = [c for c in df.columns if any(k in c.lower() for k in ['price', 'modal', 'msp', 'cost', 'rate'])]
if target_candidates:
    target_col = target_candidates[0]
else:
    # numeric કોલમમાંથી છેલ્લી
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    target_col = numeric_cols[-1]

print(f"\n🎯 Target Column (ભાવ): '{target_col}'")

# ૫. Target (y) અને Features (X)
y = pd.to_numeric(df[target_col], errors='coerce')
X_df = df.drop(columns=[target_col])

# Null વેલ્યુઝ ફરીથી ક્લીન કરો
valid_idx = y.dropna().index
y = y.loc[valid_idx]
X_df = X_df.loc[valid_idx]

# One-Hot Encoding
X = pd.get_dummies(X_df, drop_first=True)

# ૬. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ૭. Random Forest Model Training
print("\n⏳ મોડેલ ટ્રેનિંગ શરૂ થઈ રહ્યું છે...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ૮. Accuracy ચેક કરો
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n=================================")
print("🎉 મોડેલ સફળતાપૂર્વક ટ્રેન થઈ ગયું!")
print(f"📊 Accuracy (R2 Score): {r2 * 100:.2f}%")
print(f"📉 Average Error (MAE): ₹{mae:.2f}")
print("=================================")

# ૯. સેવ કરો
joblib.dump(model, '/kaggle/working/crop_price_model.pkl')
joblib.dump(X.columns.tolist(), '/kaggle/working/model_features.pkl')
print("💾 Model Saved: /kaggle/working/crop_price_model.pkl")

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# ૧. બધા CSV સ્કેન કરીને સાચો Price Dataset શોધો
valid_datasets = []

for filepath in glob.glob('/kaggle/input/**/*.csv', recursive=True):
    try:
        temp_df = pd.read_csv(filepath, nrows=10)
        # જો ફાઈલમાં ઓછામાં ઓછી ૧ numeric કોલમ હોય અને ૧૦૦+ bytes ની હોય
        numeric_cols = temp_df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) > 0 and 'disease' not in filepath.lower():
            valid_datasets.append((filepath, numeric_cols, temp_df.columns.tolist()))
    except Exception:
        continue

print("=== 🌾 મળેલા માન્ય Datasets ===")
for idx, (path, num_cols, all_cols) in enumerate(valid_datasets):
    print(f"[{idx}] {os.path.basename(path)}")
    print(f"    Columns: {all_cols[:6]}...")

# ૨. Price/Mandi વાળી શ્રેષ્ઠ ફાઈલ પસંદ કરો
best_file = None
for path, num_cols, all_cols in valid_datasets:
    cols_str = " ".join(all_cols).lower()
    if any(k in cols_str for k in ['price', 'modal', 'mandi', 'market', 'msp', 'cost']):
        best_file = path
        break

if not best_file and valid_datasets:
    best_file = valid_datasets[0][0]

print(f"\n📂 પસંદ કરેલ ડેટાસેટ: {best_file}")
df = pd.read_csv(best_file)
print(f"📊 Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(df.head())

# ૩. Missing Data સાફ કરો
df = df.dropna()

# ૪. Target Price Column નક્કી કરો
price_candidates = [c for c in df.select_dtypes(include=[np.number]).columns if any(k in c.lower() for k in ['price', 'modal', 'msp', 'cost', 'rate', 'target'])]
if price_candidates:
    target_col = price_candidates[0]
else:
    target_col = df.select_dtypes(include=[np.number]).columns[-1]

print(f"\n🎯 Target Column (જે પ્રેડિક્ટ થશે): '{target_col}'")

# ૫. X અને y અલગ કરો
y = df[target_col]
X_raw = df.drop(columns=[target_col])

# One-Hot Encoding (ટેક્સ્ટને નંબરમાં ફેરવો)
X = pd.get_dummies(X_raw, drop_first=True)

# ૬. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ૭. Machine Learning Model Training
print("\n⏳ મોડેલ ટ્રેનિંગ ચાલુ છે...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ૮. Accuracy અને Error રિઝલ્ટ
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n=================================")
print("🎉 Model Training Complete!")
print(f"📊 Accuracy (R2 Score): {r2 * 100:.2f}%")
print(f"📉 Average Error (MAE): ₹{mae:.2f}")
print("=================================")

# ૯. Model Save કરો
joblib.dump(model, '/kaggle/working/crop_price_model.pkl')
joblib.dump(X.columns.tolist(), '/kaggle/working/model_features.pkl')
print("💾 ફાઈલ સેવ થઈ ગઈ: /kaggle/working/crop_price_model.pkl")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

# ૧. ભાવને ૩ કેટેગરીમાં વહેંચો (Low, Medium, High)
labels = ['Low Price (ઓછો)', 'Medium Price (સામાન્ય)', 'High Price (વધુ)']
y_binned = pd.qcut(y, q=3, labels=labels)

# ૨. Train / Test Split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_binned, test_size=0.2, random_state=42)

# ૩. Classifier Model Train કરો
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_c, y_train_c)

# ૪. Predictions અને Confusion Matrix ગણતરી
y_pred_c = clf.predict(X_test_c)
cm = confusion_matrix(y_test_c, y_pred_c, labels=labels)

# ૫. સુંદર રંગીન Heatmap (Confusion Matrix) દોરો
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, 
            yticklabels=labels,
            cbar=True)

plt.title('🌾 Crop Price - Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('AI એ ધારેલું (Predicted Label)', fontsize=12)
plt.ylabel('સાચું પરિણામ (Actual Label)', fontsize=12)
plt.xticks(rotation=15)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ૬. Classification Report (Accuracy, Precision, Recall)
print("\n📊 Classification Report:")
print(classification_report(y_test_c, y_pred_c, target_names=labels))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

# ૧. Labels English માં રાખો જેથી ફોન્ટ વોર્નિંગ ન આવે
labels = ['Low Price', 'Medium Price', 'High Price']
y_binned = pd.qcut(y, q=3, labels=labels)

# ૨. Train / Test Split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_binned, test_size=0.2, random_state=42)

# ૩. Classifier Model Train કરો
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_c, y_train_c)

# ૪. Predictions & Confusion Matrix
y_pred_c = clf.predict(X_test_c)
cm = confusion_matrix(y_test_c, y_pred_c, labels=labels)

# ૫. Clean Heatmap Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, 
            yticklabels=labels,
            cbar=True)

plt.title('Crop Price - Confusion Matrix Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label (Model Prediction)', fontsize=12)
plt.ylabel('Actual Label (Real Data)', fontsize=12)
plt.tight_layout()
plt.show()

# ૬. Classification Report
print("=== Classification Metrics ===")
print(classification_report(y_test_c, y_pred_c, target_names=labels))

In [ ]:
# ટેસ્ટિંગ માટે એક સેમ્પલ લઈએ
sample_data = X_test.iloc[[0]]
predicted_price = model.predict(sample_data)[0]
actual_price = y_test.iloc[0]

print("========================================")
print(f"🌾 વાસ્તવિક ભાવ (Actual Price)    : ₹{actual_price:.2f}")
print(f"🤖 AI એ ધારેલો ભાવ (Predicted Price): ₹{predicted_price:.2f}")
print(f"📉 તફાવત (Difference)            : ₹{abs(actual_price - predicted_price):.2f}")
print("========================================")

In [ ]:
# Top 10 સૌથી મહત્વના Features
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = feature_importance.nlargest(10)

plt.figure(figsize=(10, 5))
top_features.plot(kind='barh', color='teal')
plt.title('Top 10 Factors Influencing Crop Prices')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
pd as pandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Top 10 સૌથી મહત્વના Features નો ગ્રાફ
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = feature_importance.nlargest(10)

plt.figure(figsize=(10, 5))
top_features.plot(kind='barh', color='teal')
plt.title('Top 10 Factors Influencing Crop Prices')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# ૧. સાચો ડેટાસેટ શોધો અને લોડ કરો
valid_csvs = [f for f in glob.glob('/kaggle/input/**/*.csv', recursive=True) if 'bharatbench' not in f.lower()]
df = pd.read_csv(valid_csvs[0]).dropna()

# ૨. Numeric Target (Price) નક્કી કરો
price_candidates = [c for c in df.select_dtypes(include=[np.number]).columns if any(k in c.lower() for k in ['price', 'modal', 'msp', 'cost'])]
target_col = price_candidates[0] if price_candidates else df.select_dtypes(include=[np.number]).columns[-1]

# ૩. Data Preprocessing
y = df[target_col]
X = pd.get_dummies(df.drop(columns=[target_col]), drop_first=True)

# ૪. Model Training
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X, y)

# ૫. Top 10 Features નો સુંદર Graph
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = feature_importance.nlargest(10)

plt.figure(figsize=(10, 5))
top_features.plot(kind='barh', color='teal')
plt.title('Top 10 Factors Influencing Crop Prices', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# ૧. બધા Datasets માંથી સાચી Price વાળી ફાઈલ શોધો
target_file = None
target_col = None

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.csv'):
            full_path = os.path.join(root, file)
            try:
                temp = pd.read_csv(full_path, nrows=5)
                # પાકના ભાવો વાળી કોલમ શોધો
                cols = [c for c in temp.columns if any(k in c.lower() for k in ['modal_price', 'price', 'modal', 'msp', 'min_price', 'priceperquintal'])]
                if cols and len(temp.select_dtypes(include=[np.number]).columns) > 0:
                    target_file = full_path
                    target_col = cols[0]
                    break
            except Exception:
                continue
    if target_file:
        break

# જો ઉપરથી ન મળે તો જેમાં સૌથી વધુ Numeric Columns હોય તે ફાઈલ લો
if not target_file:
    for f in glob.glob('/kaggle/input/**/*.csv', recursive=True):
        try:
            temp = pd.read_csv(f, nrows=5)
            num_cols = temp.select_dtypes(include=[np.number]).columns
            if len(num_cols) >= 2:
                target_file = f
                target_col = num_cols[-1]
                break
        except Exception:
            continue

print(f"📂 સાચો ડેટાસેટ મળ્યો : {target_file}")
print(f"🎯 ભાવ વાળી કોલમ     : '{target_col}'")

# ૨. પૂરો ડેટા લોડ કરો
df = pd.read_csv(target_file).dropna()

# ૩. Data Preprocessing
# Target Column ને Number માં કન્વર્ટ કરો
y = pd.to_numeric(df[target_col], errors='coerce')
valid_rows = y.dropna().index
y = y.loc[valid_rows]
X_raw = df.drop(columns=[target_col]).loc[valid_rows]

# Categorical ડેટાને Numbers માં ફેરવો
X = pd.get_dummies(X_raw, drop_first=True)

# ૪. મોડેલ ટ્રેન કરો
print("\n⏳ મોડેલ ટ્રેનિંગ શરૂ થઈ રહ્યું છે...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X, y)
print("✅ મોડેલ ટ્રેન થઈ ગયું!")

# ૫. Top 10 Factors નો Graph
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = feature_importance.nlargest(10)

plt.figure(figsize=(10, 5))
top_features.plot(kind='barh', color='teal')
plt.title('Top 10 Factors Influencing Crop Prices', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# મોડેલ અને ફિચર્સ સેવ કરો
joblib.dump(model, '/kaggle/working/crop_price_model.pkl')
joblib.dump(X.columns.tolist(), '/kaggle/working/model_features.pkl')

print("💾 મોડેલ સફળતાપૂર્વક સેવ થઈ ગયું: crop_price_model.pkl")

In [ ]:
# ડેટાસેટમાંથી સેમ્પલ લઈને ભાવ ચેક કરો
sample = X.iloc[[0]]
predicted_val = model.predict(sample)[0]
actual_val = y.iloc[0]

print("========================================")
print(f"🌾 વાસ્તવિક ભાવ (Actual Price)    : ₹{actual_val:.2f}")
print(f"🤖 AI એ ધારેલો ભાવ (Predicted Price): ₹{predicted_val:.2f}")
print("========================================")